# Baseline detection and tracking on a clip (Colab)

Runs the Phase 1 baseline: pretrained YOLO (COCO `person` / `sports ball`) + the IoU tracker + possession, on one clip from Google Drive. Its accuracy on league footage is **not known yet**; this produces outputs to inspect and label against.

1. **Runtime > Change runtime type > T4 GPU**
2. Put a clip in Drive at `MyDrive/basketball-ai-data/clips/<gym-id>/<clip>.mp4`
3. Edit the settings cell, then **Runtime > Run all**

Outputs go to `MyDrive/basketball-ai-data/outputs/<gameId>/` (never into git). Clear outputs before saving this notebook back to GitHub.

In [ ]:
# Settings
CLIP = "clips/gym-a/example.mp4"  # path inside basketball-ai-data
GAME_ID = "example-game"
START, END = 0, 30  # seconds of the clip to process
EVERY_N = 2  # process every n-th frame (2 = half the frames)

In [ ]:
# Code and dependencies
import os

if not os.path.isdir("/content/basketball-ai"):
    !git clone -q https://github.com/johnBaloran/basketball-ai /content/basketball-ai
%cd /content/basketball-ai
!git pull -q
!pip install -q -e ".[cv]"
!bball-doctor

In [ ]:
# Data folder on Google Drive
from google.colab import drive

drive.mount("/content/drive")
os.environ["BBALL_DATA_DIR"] = "/content/drive/MyDrive/basketball-ai-data"
!ls "$BBALL_DATA_DIR"

In [ ]:
# Manifest for this clip (synthetic teams; real rosters come from the league app later)
import json
from pathlib import Path

manifest = {
    "gameId": GAME_ID,
    "homeTeam": {"id": "home", "name": "Home"},
    "awayTeam": {"id": "away", "name": "Away"},
    "roster": [],
    "video": {"source": CLIP, "startSeconds": START, "endSeconds": END},
}
path = Path("/content/manifest.json")
path.write_text(json.dumps(manifest))
!bball-run /content/manifest.json --every-n {EVERY_N}

In [ ]:
# Look at the result
from IPython.display import Video

out = Path(os.environ["BBALL_DATA_DIR"]) / "outputs" / GAME_ID
print(json.dumps(json.loads((out / "possession.json").read_text()), indent=2)[:2000])
Video(str(out / "annotated.mp4"), embed=True, width=720)